# Visão Computacional com CNNs e Transformers
## 🎓 Faculdade Infnet — Pós-Graduação em Inteligência Artificial & Machine Learning
### Aula 3: O Ecossistema Hugging Face — Inferência e Fine-Tuning do BERT para Problemas de Mercado

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/allanspadini/curso-vision-transformers-infnet/blob/main/aula_03_fine_tuning_bert/aula_03_bert_fine_tuning.ipynb)

---

### 🎯 Metodologia Pedagógica: Situação-Problema ➔ Solução de Engenharia ➔ Aplicação de Mercado
Neste laboratório, entramos no ecossistema padrão da indústria para modelos fundacionais de NLP e Visão: a biblioteca **`transformers` da Hugging Face**.

Abordaremos:
1. **Inferência Off-The-Shelf**: Como utilizar modelos pré-treinados para inferência imediata via `pipeline` e via forward pass de baixo nível no PyTorch.
2. **Fine-Tuning para Análise de Sentimentos (Sequence Classification)**: Adaptação do BERT para classificar avaliações e opiniões de mercado (positivo/negativo).
3. **Fine-Tuning para Reconhecimento de Entidades Nomeadas (Token Classification / NER)**: Extração de informações críticas (Pessoas, Organizações, Locais) com o desafio de engenharia do **alinhamento de subwords WordPiece** (`-100` masking).


### 1. Configuração do Ambiente e Autenticação no Hugging Face Hub

Instalamos as dependências do ecossistema Hugging Face e carregamos o token do arquivo `.env` (quando disponível) para acesso a repositórios autenticados.


In [ ]:
!pip install -q transformers datasets evaluate accelerate seqeval scikit-learn python-dotenv tqdm

import os
import random
import numpy as np
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from dotenv import load_dotenv

# Carregar variáveis de ambiente do .env
load_dotenv()
hf_token = os.getenv("HF_TOKEN") or os.getenv("HF_Token")

if hf_token:
    try:
        from huggingface_hub import login
        login(token=hf_token)
        print("🔑 Autenticado no Hugging Face Hub com sucesso!")
    except Exception as e:
        print(f"⚠️ Aviso ao autenticar no Hugging Face Hub: {e}")
else:
    print("ℹ️ HF_TOKEN não encontrado no ambiente. Usando acesso público padrão do Hugging Face Hub.")

# Reprodutibilidade e dispositivo
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔥 Dispositivo de Execução: {device}")


### 2. Inferência Off-The-Shelf com Modelos Pré-Treinados

No ecossistema Hugging Face, existem dois caminhos principais para inferência:
1. **`pipeline()` (Alto Nível)**: Ideal para prototipação rápida e pipelines de produção que exigem zero boilerplate.
2. **`AutoTokenizer` + `AutoModel` (Baixo Nível / PyTorch Puro)**: Dá controle total sobre tensores, máscaras e logits de saída.


In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

# 1. Inferência de Alto Nível com pipeline()
print("⚡ --- 1. Inferência com pipeline() ---")
sentiment_pipe = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english", device=device)
ner_pipe = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple", device=device)

market_reviews = [
    "The new cloud database service reduced our operational latency by 40%.",
    "Customer service was unresponsive and the checkout system crashed repeatedly."
]

for review in market_reviews:
    res = sentiment_pipe(review)[0]
    print(f"📝 Texto: '{review}'")
    print(f"   Sentimento: {res['label']} (Confiança: {res['score']:.4f})\n")

market_news = "Sundar Pichai announced that Google will invest 2 billion dollars in a new data center in Texas."
ner_entities = ner_pipe(market_news)
print(f"📰 Notícia de Mercado: '{market_news}'")
print("🏷️ Entidades Extraídas:")
for ent in ner_entities:
    print(f"   • {ent['word']} ➔ {ent['entity_group']} (Confiança: {ent['score']:.4f})")

# 2. Inferência em Baixo Nível via PyTorch (Tensor Inspection)
print("\n🔬 --- 2. Inspeção de Tensores por Baixo dos Panos ---")
tokenizer_bert = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
model_bert = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english").to(device)

sample_text = "This machine learning platform is exceptionally fast."
inputs = tokenizer_bert(sample_text, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model_bert(**inputs)
    logits = outputs.logits
    probabilities = F.softmax(logits, dim=-1)

print(f"Tokens de Entrada (IDs): {inputs['input_ids'].tolist()}")
print(f"Subwords Decodificadas:  {tokenizer_bert.convert_ids_to_tokens(inputs['input_ids'][0])}")
print(f"Logits de Saída:         {logits.cpu().numpy()[0]}")
print(f"Probabilidade Negativo:  {probabilities[0][0].item():.4f} | Probabilidade Positivo: {probabilities[0][1].item():.4f}")


### 3. Fine-Tuning do BERT para Análise de Sentimentos no Mercado

#### A Situação-Problema do Mundo Real:
Modelos off-the-shelf genéricos muitas vezes não capturam o jargão específico de um domínio comercial (ex: avaliações de e-commerce, suporte técnico, feedback de produtos).  
A solução padrão é realizar o **Fine-Tuning supervisionado** do `bert-base-uncased` com uma cabeça de classificação (`AutoModelForSequenceClassification`).

Utilizaremos o dataset `cornell-movie-review-data/rotten_tomatoes` e a API padrão de mercado **`Trainer`** do Hugging Face.


In [ ]:
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import evaluate

# 1. Carregamento do Dataset
dataset = load_dataset("cornell-movie-review-data/rotten_tomatoes")

# Subconjuntos representativos para execução ágil (ajustáveis conforme GPU disponível)
train_size = 800 if torch.cuda.is_available() else 200
eval_size = 200 if torch.cuda.is_available() else 50

train_data_sent = dataset["train"].select(range(train_size))
eval_data_sent = dataset["validation"].select(range(eval_size))
print(f"📊 Amostras de Treino: {len(train_data_sent)} | Validação: {len(eval_data_sent)}")

# 2. Tokenização em Lote com Truncamento
tokenizer_clf = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

def preprocess_sentiment(examples):
    return tokenizer_clf(examples["text"], truncation=True, max_length=128)

encoded_train_sent = train_data_sent.map(preprocess_sentiment, batched=True)
encoded_eval_sent = eval_data_sent.map(preprocess_sentiment, batched=True)

# 3. Métrica de Avaliação (Accuracy)
accuracy_metric = evaluate.load("accuracy")

def compute_metrics_sentiment(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)

# 4. Inicialização do Modelo BERT com Cabeça de Classificação Linear
model_sentiment = AutoModelForSequenceClassification.from_pretrained(
    "google-bert/bert-base-uncased",
    num_labels=2
)

# 5. Configuração do Treinamento
training_args_sent = TrainingArguments(
    output_dir="./results_bert_sentiment",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=10,
    report_to="none"
)

trainer_sent = Trainer(
    model=model_sentiment,
    args=training_args_sent,
    train_dataset=encoded_train_sent,
    eval_dataset=encoded_eval_sent,
    processing_class=tokenizer_clf,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer_clf),
    compute_metrics=compute_metrics_sentiment
)

print("🚀 Iniciando Fine-Tuning do BERT para Análise de Sentimentos...")
trainer_sent.train()
eval_results = trainer_sent.evaluate()
print(f"✅ Avaliação Final -> Acurácia: {eval_results['eval_accuracy']:.4f}")


In [ ]:
# Testes de Predição em Frases do Mundo Real Pós-Fine-Tuning
model_sentiment.eval()

customer_feedback = [
    "The delivery arrived earlier than expected and the quality is outstanding!",
    "Terrible customer service, the device stopped working after just two days.",
    "Very average experience, nothing really stood out."
]

print("🔍 --- Inferência Pós-Fine-Tuning (Classificação de Sentimentos) ---")
for text in customer_feedback:
    inputs = tokenizer_clf(text, return_tensors="pt", truncation=True, max_length=128).to(model_sentiment.device)
    with torch.no_grad():
        logits = model_sentiment(**inputs).logits
        prob = F.softmax(logits, dim=-1)[0]
        pred_label = "POSITIVO" if prob[1] > prob[0] else "NEGATIVO"
        confidence = prob[prob.argmax()].item()
    print(f"💬 Comentário: '{text}'")
    print(f"   Sentimento Previsto: {pred_label} (Confiança: {confidence:.4f})\n")


### 4. Fine-Tuning do BERT para Reconhecimento de Entidades Nomeadas (NER)

#### O Desafio Crítico de Engenharia: Alinhamento de Subwords (`word_ids`)
No problema de **Token Classification** (NER), cada palavra possui um rótulo (ex: `B-PER`, `I-PER`, `B-ORG`, `O`).  
Porém, o tokenizador WordPiece do BERT divide palavras complexas em múltiplas sub-unidades (ex: *'Washington'* $\to$ `['Wash', '##ington']`).

Se a palavra original tinha o rótulo `B-LOC`:
* A primeira sub-unidade (`Wash`) recebe `B-LOC`.
* As sub-unidades subsequentes (`##ington`) e tokens especiais (`[CLS]`, `[SEP]`, `[PAD]`) devem receber o rótulo **`-100`**.  
* No PyTorch, `nn.CrossEntropyLoss(ignore_index=-100)` zera automaticamente o gradiente dessas posições, impedindo que a fragmentação de palavras corrompa o aprendizado do modelo!


In [ ]:
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification

# 1. Carregamento do Benchmark da Indústria (CoNLL-2003)
conll_dataset = load_dataset("lhoestq/conll2003")

# Mapeamento de Rótulos
label_list = [
    "O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"
]
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

tokenizer_ner = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

# 2. Função Crucial de Engenharia: Alinhamento de Subwords com -100
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer_ner(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=64
    )
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                # [CLS], [SEP] e [PAD] recebem -100 (ignorado no gradiente)
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # Primeira subword da palavra: recebe o rótulo real
                label_ids.append(label[word_idx])
            else:
                # Subwords subsequentes: mascaradas com -100
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Subconjuntos concisos para treino ágil
ner_train_size = 500 if torch.cuda.is_available() else 150
ner_eval_size = 150 if torch.cuda.is_available() else 40

train_conll = conll_dataset["train"].select(range(ner_train_size))
eval_conll = conll_dataset["validation"].select(range(ner_eval_size))

encoded_train_ner = train_conll.map(tokenize_and_align_labels, batched=True)
encoded_eval_ner = eval_conll.map(tokenize_and_align_labels, batched=True)

# 3. Métricas com Seqeval
seqeval_metric = evaluate.load("seqeval")

def compute_metrics_ner(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results = seqeval_metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"]
    }

# 4. Inicialização do Modelo para Token Classification
model_ner = AutoModelForTokenClassification.from_pretrained(
    "google-bert/bert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# 5. Configuração e Treinamento
training_args_ner = TrainingArguments(
    output_dir="./results_bert_ner",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    learning_rate=3e-5,
    weight_decay=0.01,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=10,
    report_to="none"
)

trainer_ner = Trainer(
    model=model_ner,
    args=training_args_ner,
    train_dataset=encoded_train_ner,
    eval_dataset=encoded_eval_ner,
    processing_class=tokenizer_ner,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer_ner),
    compute_metrics=compute_metrics_ner
)

print("🚀 Iniciando Fine-Tuning do BERT para Reconhecimento de Entidades (NER)...")
trainer_ner.train()
print("🎉 Fine-Tuning de NER concluído com sucesso!")


In [ ]:
# Teste Prático de Extração de Entidades em Textos Corporativos
model_ner.eval()

business_text = "Satya Nadella spoke at Microsoft headquarters in Redmond Washington about artificial intelligence."
inputs_ner = tokenizer_ner(business_text, return_tensors="pt").to(model_ner.device)

with torch.no_grad():
    outputs_ner = model_ner(**inputs_ner)
    predictions_ner = outputs_ner.logits.argmax(dim=-1)[0].cpu().numpy()

tokens_text = tokenizer_ner.convert_ids_to_tokens(inputs_ner["input_ids"][0])

print("🔍 --- Entidades Extraídas pelo BERT Fine-Tuned ---")
for token, pred_id in zip(tokens_text, predictions_ner):
    tag = id2label[pred_id]
    if tag != "O" and token not in ["[CLS]", "[SEP]"]:
        print(f"   Token: {token:12s} ➔ Rótulo: {tag}")


### 5. Desafios Práticos de Fixação (Hands-on para Casa)

1. **Desafio 1 (BERTimbau em Português)**: Substitua o modelo base por `neuralmind/bert-base-portuguese-cased` e realize o fine-tuning em um corpus de sentimento ou NER em língua portuguesa (ex: dataset `tweet_eval` ou notícias em português).
2. **Desafio 2 (Fine-Tuning com LoRA / PEFT)**: Utilize a biblioteca `peft` para congelar 99% dos pesos do BERT e treinar apenas matrizes de adaptação de baixo posto (LoRA) com rank $r=8$. Compare a memória VRAM gasta e o tempo por época!
3. **Desafio 3 (Extração de Embeddings Semânticos)**: Utilize `AutoModel.from_pretrained("google-bert/bert-base-uncased")` e calcule o *Mean Pooling* da última camada oculta (`last_hidden_state`) para calcular a similaridade de cosseno entre pares de frases.
